In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

# After scraping and splitting

In [6]:
import pickle

with open('../data/dataset.pkl', 'rb') as f:
    data = pickle.load(f)

In [7]:

import re

def extract_commit_hash(input_string):
    """
    Extract the commit hash from any string.

    Args:
        input_string (str): The input string to search for a commit hash.

    Returns:
        str or None: The first valid commit hash if found, otherwise None.
    """
    pattern = r"\b[a-fA-F0-9]{40}\b"  # Regex for a 40-character hexadecimal hash
    match = re.search(pattern, input_string)
    return match.group(0) if match else None

In [8]:
# make hf dataset of data

from datasets import Dataset

records = []
for k, v in data.items():
    record = {
        "cve": k,
        "published_date": v["published_date"],
        "desc": v["desc"],
        "commit_urls": list(set([commit for commit in v["ground_truth"]["commit"] ])),
        "commits": list(set([
                            hash_ for commit in v["ground_truth"]["commit"]
                            if (hash_ := extract_commit_hash(commit)) is not None
                        ])),
    }
    records.append(record)

ds = Dataset.from_list(records)


In [ ]:
from datasets import load_dataset

ds_patches = load_dataset("andstor/cvevc_commits", name="patches")

In [12]:
from datasets import DatasetDict

ddict = DatasetDict({
    "train": ds.filter(lambda x: any([commit in ds_patches["train"]["commit_id"] for commit in x["commits"]]), batch_size=1, num_proc=10),
    "test": ds.filter(lambda x: any([commit in ds_patches["test"]["commit_id"] for commit in x["commits"]]), batch_size=1, num_proc=10),
    "validation": ds.filter(lambda x: any([commit in ds_patches["validation"]["commit_id"] for commit in x["commits"]]), batch_size=1, num_proc=10)
})

Filter (num_proc=10):   0%|          | 0/19256 [00:00<?, ? examples/s]

Filter (num_proc=10): 100%|██████████| 19256/19256 [00:04<00:00, 4554.57 examples/s]


In [ ]:
ddict.push_to_hub("andstor/cvevc_cve", private=False, max_shard_size="250MB")

Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.81s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/andstor/cvevc_cve/commit/f96011d6b503637282c349d3c5dc5874d33bc77f', commit_message='Upload dataset', commit_description='', oid='f96011d6b503637282c349d3c5dc5874d33bc77f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/andstor/cvevc_cve', endpoint='https://huggingface.co', repo_type='dataset', repo_id='andstor/cvevc_cve'), pr_revision=None, pr_num=None)